In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score, brier_score_loss, log_loss
)

SEED = 42
np.random.seed(SEED)
n_records = 3000

# 1. Synthetic Data Generation (Simulation Benchmark)
patient_ids = [f"PHC-EB-{2000 + i}" for i in range(n_records)]
age = np.random.randint(0, 75, size=n_records)
gender = np.random.choice(["Female", "Male"], size=n_records, p=[0.62, 0.38])
lga = np.random.choice(["Abakaliki", "Ebonyi", "Izzi", "Ikwo", "Afikpo North"], size=n_records)
clinic_type = np.random.choice(
    ["Antenatal Care", "Routine Immunization", "General Outpatient", "Chronic Disease (HTN/DM)"],
    size=n_records,
    p=[0.32, 0.28, 0.25, 0.15]
)
hypertension = np.where(age >= 35, np.random.binomial(1, 0.28, size=n_records), 0)
diabetes = np.where(age >= 35, np.random.binomial(1, 0.12, size=n_records), 0)
lead_time_days = np.random.geometric(p=0.12, size=n_records) - 1
past_no_shows = np.random.poisson(lam=0.65, size=n_records)
distance_to_phc_km = np.round(np.random.exponential(scale=4.5, size=n_records) + 0.5, 1)

# Latent no-show probability prior to any SMS intervention
logit_baseline = (
    -1.1
    + 0.06 * lead_time_days
    + 0.08 * distance_to_phc_km
    + 0.45 * past_no_shows
    - 0.30 * (clinic_type == "Routine Immunization").astype(int)
    + 0.25 * (age < 22).astype(int)
)
prob_baseline = 1 / (1 + np.exp(-logit_baseline))
no_show = np.random.binomial(1, prob_baseline)

df = pd.DataFrame({
    'PatientID': patient_ids, 'Age': age, 'Gender': gender, 'LGA': lga,
    'ClinicType': clinic_type, 'Hypertension': hypertension, 'Diabetes': diabetes,
    'Distance_KM': distance_to_phc_km, 'LeadTimeDays': lead_time_days,
    'PastNoShows': past_no_shows, 'NoShow': no_show
})
df.to_csv('phc_noshow_synthetic.csv', index=False)

# 2. Dataset Overview
overall_no_show = df['NoShow'].mean()
print("=" * 65)
print("1. SIMULATION COHORT SUMMARY (METHODOLOGICAL BENCHMARK)")
print("=" * 65)
print(f"Total Synthetic Records:       {df.shape[0]}")
print(f"Overall Baseline No-Show Rate: {overall_no_show * 100:.2f}%")
print("Note: Data generated under defined probabilistic equations for pipeline benchmarking.")

# EDA Figure
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
clinic_agg = df.groupby('ClinicType')['NoShow'].mean().sort_values()
axes[0].barh(clinic_agg.index, clinic_agg.values, color='#1f77b4', height=0.5)
axes[0].set_title('Synthetic No-Show Rate by Clinic Type')
axes[0].set_xlabel('Proportion No-Show')

axes[1].boxplot(
    [df[df['NoShow'] == 0]['LeadTimeDays'], df[df['NoShow'] == 1]['LeadTimeDays']],
    labels=['Attended (0)', 'No-Show (1)'],
    showfliers=False
)
axes[1].set_title('Booking Lead Time (Days) Distribution')
axes[1].set_ylabel('Days')
plt.tight_layout()
plt.savefig('eda_summary.png', dpi=300)
plt.close()

# 3. Leakage-Safe Data Preparation
# Prediction Point: At scheduling. SMSReceived is excluded to prevent circular target intervention leakage.
X = df.drop(columns=['PatientID', 'NoShow'])
y = df['NoShow']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

num_cols = ['Age', 'Distance_KM', 'LeadTimeDays', 'PastNoShows']
cat_cols = ['Gender', 'LGA', 'ClinicType', 'Hypertension', 'Diabetes']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)

# 4. Pipeline Definition
dummy = Pipeline([('prep', preprocessor), ('model', DummyClassifier(strategy='stratified', random_state=SEED))])
pipe_lr = Pipeline([('prep', preprocessor), ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED))])
pipe_rf = Pipeline([('prep', preprocessor), ('model', RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_leaf=4, class_weight='balanced', random_state=SEED))])

# 5. 5-Fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = ['roc_auc', 'average_precision', 'f1']

cv_dummy = cross_validate(dummy, X_train, y_train, cv=cv, scoring=scoring)
cv_lr = cross_validate(pipe_lr, X_train, y_train, cv=cv, scoring=scoring)
cv_rf = cross_validate(pipe_rf, X_train, y_train, cv=cv, scoring=scoring)

print("\n" + "=" * 65)
print("2. 5-FOLD STRATIFIED CROSS-VALIDATION (TRAIN SET: N=2,400)")
print("=" * 65)
print(f"{'Model':<22}{'ROC-AUC (Mean ± SD)':<22}{'PR-AUC (Mean ± SD)':<20}{'F1 (Mean ± SD)'}")
print("-" * 65)
print(f"Dummy (Baseline)      {cv_dummy['test_roc_auc'].mean():.4f} ± {cv_dummy['test_roc_auc'].std():.4f}      {cv_dummy['test_average_precision'].mean():.4f} ± {cv_dummy['test_average_precision'].std():.4f}    {cv_dummy['test_f1'].mean():.4f} ± {cv_dummy['test_f1'].std():.4f}")
print(f"Logistic Regression   {cv_lr['test_roc_auc'].mean():.4f} ± {cv_lr['test_roc_auc'].std():.4f}      {cv_lr['test_average_precision'].mean():.4f} ± {cv_lr['test_average_precision'].std():.4f}    {cv_lr['test_f1'].mean():.4f} ± {cv_lr['test_f1'].std():.4f}")
print(f"Random Forest         {cv_rf['test_roc_auc'].mean():.4f} ± {cv_rf['test_roc_auc'].std():.4f}      {cv_rf['test_average_precision'].mean():.4f} ± {cv_rf['test_average_precision'].std():.4f}    {cv_rf['test_f1'].mean():.4f} ± {cv_rf['test_f1'].std():.4f}")

# 6. Held-Out Test Evaluation (N=600)
pipe_lr.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)

y_prob_lr = pipe_lr.predict_proba(X_test)[:, 1]
y_prob_rf = pipe_rf.predict_proba(X_test)[:, 1]
y_pred_rf = pipe_rf.predict(X_test)

print("\n" + "=" * 65)
print("3. HELD-OUT TEST PERFORMANCE COMPARISON (N=600)")
print("=" * 65)
print(f"{'Metric':<20}{'Logistic Regression':<24}{'Random Forest'}")
print("-" * 65)
print(f"Test ROC-AUC        {roc_auc_score(y_test, y_prob_lr):<24.4f}{roc_auc_score(y_test, y_prob_rf):.4f}")
print(f"Test PR-AUC         {average_precision_score(y_test, y_prob_lr):<24.4f}{average_precision_score(y_test, y_prob_rf):.4f}")
print(f"Brier Score (Cal.)  {brier_score_loss(y_test, y_prob_lr):<24.4f}{brier_score_loss(y_test, y_prob_rf):.4f}")
print(f"Log-Loss            {log_loss(y_test, y_prob_lr):<24.4f}{log_loss(y_test, y_prob_rf):.4f}")

# 7. Operational Threshold Optimization (Targeting High Recall for Public Health)
print("\n" + "=" * 65)
print("4. OPERATIONAL THRESHOLD OPTIMIZATION (RANDOM FOREST)")
print("=" * 65)
print("Objective: Maximize recall to capture missed appointments for follow-up.")
print(f"{'Threshold':<12}{'Precision':<14}{'Recall':<14}{'F1-Score':<14}{'Operational Status'}")
print("-" * 65)
for t in [0.30, 0.35, 0.40, 0.45, 0.50, 0.60]:
    preds = (y_prob_rf >= t).astype(int)
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)
    status = "Recommended Operating Point" if t == 0.40 else ("Default Uncalibrated" if t == 0.50 else "")
    print(f"{t:<12.2f}{p:<14.4f}{r:<14.4f}{f:<14.4f}{status}")

# 8. Permutation Feature Importance on Held-Out Test Set
perm_res = permutation_importance(pipe_rf, X_test, y_test, n_repeats=10, random_state=SEED, scoring='roc_auc')
perm_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance_Mean': perm_res.importances_mean,
    'Importance_Std': perm_res.importances_std
}).sort_values(by='Importance_Mean', ascending=False)

print("\n" + "=" * 65)
print("5. TEST-SET PERMUTATION FEATURE IMPORTANCE (ROC-AUC DROP)")
print("=" * 65)
print(perm_df.to_string(index=False))

plt.figure(figsize=(9, 4.5))
top_perm = perm_df.sort_values(by='Importance_Mean', ascending=True)
plt.barh(top_perm['Feature'], top_perm['Importance_Mean'], xerr=top_perm['Importance_Std'], color='#2ca02c', height=0.5)
plt.title('Permutation Feature Importance on Held-Out Test Set (Metric: ROC-AUC)')
plt.xlabel('Mean Drop in ROC-AUC')
plt.tight_layout()
plt.savefig('model_feature_importance.png', dpi=300)
plt.close()
print("\nAll pipeline evaluations complete. Figures saved.")

1. SIMULATION COHORT SUMMARY (METHODOLOGICAL BENCHMARK)
Total Synthetic Records:       3000
Overall Baseline No-Show Rate: 52.97%
Note: Data generated under defined probabilistic equations for pipeline benchmarking.


/tmp/ipykernel_590/1852316059.py:74: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[1].boxplot(



2. 5-FOLD STRATIFIED CROSS-VALIDATION (TRAIN SET: N=2,400)
Model                 ROC-AUC (Mean ± SD)   PR-AUC (Mean ± SD)  F1 (Mean ± SD)
-----------------------------------------------------------------
Dummy (Baseline)      0.5119 ± 0.0174      0.5359 ± 0.0095    0.5383 ± 0.0165
Logistic Regression   0.6862 ± 0.0165      0.7157 ± 0.0249    0.6304 ± 0.0065
Random Forest         0.6789 ± 0.0158      0.7016 ± 0.0214    0.6270 ± 0.0104

3. HELD-OUT TEST PERFORMANCE COMPARISON (N=600)
Metric              Logistic Regression     Random Forest
-----------------------------------------------------------------
Test ROC-AUC        0.6832                  0.6709
Test PR-AUC         0.7103                  0.6804
Brier Score (Cal.)  0.2238                  0.2293
Log-Loss            0.6378                  0.6509

4. OPERATIONAL THRESHOLD OPTIMIZATION (RANDOM FOREST)
Objective: Maximize recall to capture missed appointments for follow-up.
Threshold   Precision     Recall        F1-Score      Op